# Embedding the gold-1000 labels

Embeds the 1,000 gold labels with `LabelEmbeddingService` (Qwen3-Embedding-0.6B, three
servers) under the `label_clustering` instruction. This is the first step towards
choosing a representative evaluation subset by similarity instead of at random.

**Which passages.** Label set `gold1000-r4`: the `gold1000-r1` ids, in its order, each with
its renderer-4 passage from `full214780-r4`, the cleanest rendering there is: renderer 4
cuts the checklist boilerplate it knows, which would otherwise make unrelated labels look
alike. It does not cut all of it -- storage and seal-check lines such as "store in dry
place at room temperature" still appear -- so neighbours that match only on those lines
are worth looking for below. The gold annotations stay in `gold1000-r1`, joined by
`dsld_id`.

**Where the vectors go.** Into the label set, as
`data/dsld/labels/gold1000-r4/embeddings/label_clustering-<config hash>/`: one embedding per
instruction, so a second wording can be compared without overwriting the first. Running
the cell again resumes it and embeds only what is not done.

In [ ]:
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from engine import config
from engine.repositories.dsld.labels import loader as label_sets
from engine.repositories.dsld.labels import paths as label_paths
from engine.services.label_embedding.service import LabelEmbeddingService

LABEL_SET = "gold1000-r4"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    stream=sys.stdout,
    force=True,
)

## The labels

The gold ids in their file order, each with its r4 passage.

In [ ]:
passages_sha, labels = label_sets.load_flattened(LABEL_SET)

chars = pd.Series([len(label.passage) for label in labels.values()], name="passage chars")
print(chars.describe().round(0).to_string())
len(labels), passages_sha

## Is the model here, and is the GPU free?

The code does not download the model. If the first line says it is missing, run the
`hf download` command at the top of `configs/embed.toml`. The three servers need ~5 GB
each (~16 GB together); anything listed under compute apps is holding the card.

In [5]:
model_path = Path(config.read_toml(config.EMBED_CONFIG_PATH)["model"]["model_path"])
print(f"model file {'present' if model_path.is_file() else 'MISSING'}: {model_path}")

model file present: /var/tmp/ge74wan/hf_cache/Qwen3-Embedding-0.6B-GGUF/Qwen3-Embedding-0.6B-f16.gguf


In [4]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

memory.free [MiB]
24111 MiB
pid, process_name, used_gpu_memory [MiB]


## The run

The service embeds every label of the set not yet done, writes the vectors into the
label set, and returns the embedding's directory. Leaving the `with` block stops the
servers and frees the card, also on an error or an interrupt. A label that fails is
listed in the embedding's `failed.jsonl`, not lost with the other 999.

In [ ]:
with LabelEmbeddingService() as svc:
    embedding_dir = svc.run(LABEL_SET)
name = embedding_dir.name
embedding_dir

## Read the vectors back

Rows in gold order; a failed label has no row, and its id is in `failed.jsonl`. A vector
joins its label by `dsld_id`: `labels[ids[r]]` is what `matrix[r]` was made from.

In [ ]:
ids, matrix = label_sets.load_embedding(LABEL_SET, name)
kept = ids.tolist()

failed_path = embedding_dir / label_paths.FAILED_NAME
failed = [json.loads(line) for line in failed_path.read_text().splitlines()] if failed_path.exists() else []
print(f"embedded {len(kept)}   failed {len(failed)}   {failed}")
print(label_sets.read_embedding_manifest(LABEL_SET, name)["instruction"])
matrix.shape

## A first look

The vectors are L2-normalised by the server, so cosine similarity is a dot product.
First the spread of pairwise similarity: if almost every pair sits in a narrow band,
the instruction is not separating labels much.

In [9]:
sim = matrix @ matrix.T
upper = sim[np.triu_indices_from(sim, k=1)]
print(f"norms: min {np.linalg.norm(matrix, axis=1).min():.4f}  max {np.linalg.norm(matrix, axis=1).max():.4f}")
print(pd.Series(upper, name="pairwise cosine").describe(percentiles=[.01, .1, .5, .9, .99]).round(3).to_string())

norms: min 1.0000  max 1.0000
count    499500.000
mean          0.557
std           0.060
min           0.303
1%            0.424
10%           0.483
50%           0.556
90%           0.631
99%           0.709
max           0.989


Each sampled label beside its three nearest neighbours. Neighbours that share purposes
and warnings, not just a brand or a product form, are what the instruction is for.

In [11]:
# A seeded handful of labels. Change SAMPLE_SEED to see others.
SAMPLE_SEED = 66
rng = np.random.default_rng(SAMPLE_SEED)
row_of = {i: r for r, i in enumerate(kept)}

for dsld_id in rng.choice(kept, size=5, replace=False):
    r = row_of[int(dsld_id)]
    print(f"=== {dsld_id}  {labels[dsld_id].name}")
    print("   " + labels[dsld_id].passage[:300].replace("\n", " ") + " ...")
    for n in np.argsort(-sim[r])[1:4]:
        nid = kept[n]
        print(f"  {sim[r, n]:.3f}  {nid}  {labels[nid].name}")
        print("         " + labels[nid].passage[:200].replace("\n", " ") + " ...")
    print()

=== 214081  Green Superfood The Original
   green superfood the original green superfood the original our most popular blend thoughtfully combines our alkalizing farm fresh greens and wholesome fruits and veggies with nutrient rich superfoods for a delicious way to feel amazing everyday wheat grass spirulina spinach pineapple barley grass chl ...
  0.777  45520  Grass Juice 750 mg
         grass juice vegan product organic grass juice suggested daily intake take one tablet daily warning if you are pregnant or lactating consult a health care professional before using this product gmp com ...
  0.773  184772  Spirulina & Chlorella
         spirulina chlorella two superfoods combined as one powerful supplement boosts antioxidant activity supports toxin removal science backed quality since suggested use as take two capsules one to three t ...
  0.772  58178  Green Barley Grass 500 mg
         green barley grass good n natural manufacturing corp formula no gluten no yeast no wheat no milk o